# Metrics: copy number

Computes and saves the evaluation metrics for the categorical copy-number omic (classes -2, -1, 0, 1, 2) for one
generation method and setting:

| metric | what | file |
|---|---|---|
| Sharpness | normalised Shannon entropy of the sampled classes, per datapoint | `sharpness/copynumber_sharpness*.csv` |
| Calibration | ECE of the majority-class prediction per feature (10 confidence bins) | `calibration/copynumber_calibration*.csv` |
| Scoring rule | log score, -log P(true class), per datapoint | `CRPS/copy_number_logscore*.csv` |
| Accuracy | balanced accuracy of the point prediction per feature | `RMSE/copynumber_feature_balanced_accuracy*.csv` |

File suffix: `''` latent sampling, `_MCD` MC Dropout, then `_OOD` for the held-out rows.
Shared code: `omics_metrics.py` and `omics_config.py`. Plots: `exploration_3_copynumber.ipynb`.

In [ ]:
import pandas as pd

import omics_config as cfg
import omics_metrics as om

OMIC = "copynumber"
METHOD = "latent"       # "latent" (latent sampler) | "mcd" (MC Dropout)
SETTING = "IS"          # "IS" (in-sample) | "OOD" (held-out rows)

cfg.SAVE_OUTPUTS = True        # False: compute and inspect without writing files
cfg.DEBUG_N_FEATURES = None    # e.g. 100 for a quick test on the first 100 features (never saves)

## 1. Data
Same alignment as the continuous omics (see `metrics_1_continuous.ipynb`), without standardisation. The samples
are rounded to the nearest integer class and clipped to -2..2 (a no-op for the latent in-sample cube, which is already integer).

In [ ]:
data, imputed, cube = om.prepare(OMIC, METHOD, SETTING)
print(f"{OMIC} | {METHOD} | {SETTING}")
print(f"ground truth {data.shape}, predictions {imputed.shape}, samples {cube.shape}")
print("sampled values:", sorted(pd.unique(cube.ravel()))[:10])

## 2. Sharpness: normalised entropy
Per datapoint: Shannon entropy (in bits) of the class frequencies among the 100 samples, divided by log2(5),
so 0 = all samples agree and 1 = uniform over the five classes.

In [ ]:
sharpness = om.cn_entropy(cube, imputed.index, imputed.columns)
display(sharpness.mean().describe().rename("feature-wise mean entropy"))

## 3. Calibration: ECE
Per feature: confidence = frequency of the most sampled class, accuracy = whether that class equals the ground
truth; ECE = sum over 10 confidence bins of (bin share) x |accuracy - confidence|.

In [ ]:
ece_table = om.feature_wise_ece(data, cube)
calibration = ece_table.set_index("feature")[["ECE"]]
print(f"Features with no ground truth: {(ece_table['n_observations_used'] == 0).sum()}")
print(f"Features with > 50% missing: {(ece_table['pct_missing'] > 50).sum()}")
display(ece_table.head())

## 4. Scoring rule: log score
Class probabilities are clipped at 1e-15 and renormalised before taking the log.

In [ ]:
logscore = om.logscore_matrix(data, cube)
display(logscore.mean().describe().rename("feature-wise mean log score"))

## 5. Balanced accuracy of the point prediction

In [ ]:
balanced_accuracy = om.balanced_accuracy(data, imputed)
display(balanced_accuracy.describe())

## 6. Save

In [ ]:
results = {"sharpness": sharpness, "calibration": calibration, "logscore": logscore, "balanced_accuracy": balanced_accuracy}
om.save_results(results, OMIC, METHOD, SETTING)
summary = om.summarize(results, OMIC, METHOD, SETTING)
om.append_summary(summary)
pd.Series(summary)

## 7. Optional: every method x setting

In [ ]:
RUN_ALL = False

if RUN_ALL:
    summaries = []
    for method in cfg.METHODS:
        for setting in cfg.SETTINGS:
            summaries.append(om.run(OMIC, method, setting)[1])
    display(pd.DataFrame(summaries))